In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import pynance as pn

In [ ]:
import seaborn as sea

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
import talib

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

In [ ]:
df = pd.read_csv('../data/raw_analyst_ratings.csv')

In [ ]:
df.head()

In [ ]:
df['headline'].describe()

In [ ]:
 df['headline'] = df['headline'].astype('string')


In [ ]:
df['headline'].describe()

In [ ]:
df['headline_length'] = df['headline'].astype(str).str.len()

In [ ]:
df.describe()

In [ ]:
df['headline_length'].quantile([0.1, 0.25, 0.5, 0.75, 0.9])

In [ ]:
publisher_counts = df['publisher'].value_counts()

publisher_counts.head(20)

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

articles_per_day = df.groupby(df['date'].dt.date).size()
articles_per_day.plot(figsize=(12,5))

In [ ]:

vectorizer = CountVectorizer(stop_words='english', max_df=0.8, min_df=10)
X = vectorizer.fit_transform(df['headline'])

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

In [ ]:
words = vectorizer.get_feature_names_out()

for i, topic in enumerate(lda.components_):
    top_words = [words[j] for j in topic.argsort()[-10:]]
    print(f"Topic {i}: {top_words}")

In [ ]:
df['month'] = df['date'].dt.to_period('M')
df.groupby('month').size().plot(figsize=(12,5))


In [ ]:
df['day_of_week'] = df['date'].dt.day_name()
df['day_of_week'].value_counts().plot(kind='bar')


In [ ]:
df['domain'] = df['publisher'].str.split('@').str[-1]
df['domain'].value_counts().head(20)


In [ ]:
# Simple Moving Average (20-day)
df['SMA_20'] = talib.SMA(df['Close'], timeperiod=20)

# Exponential Moving Average (50-day)
df['EMA_50'] = talib.EMA(df['Close'], timeperiod=50)

In [ ]:
df['RSI_14'] = talib.RSI(df['Close'], timeperiod=14)

In [ ]:
df['MACD'], df['MACD_signal'], df['MACD_hist'] = talib.MACD(
    df['Close'], fastperiod=12, slowperiod=26, signalperiod=9
)

In [ ]:
df['Returns'] = pn.freturn.simple(df['Close'])

df['Cum_Returns'] = (1 + df['Returns']).cumprod()
df['Volatility'] = df['Returns'].rolling(20).std()
risk_free_rate = 0
df['Sharpe'] = (df['Returns'] - risk_free_rate) / df['Volatility']

#Price + SMA + EMA

In [ ]:
plt.plot(df['Close'])
plt.plot(df['SMA_20'])
plt.plot(df['EMA_50'])
plt.title("Close Price with SMA & EMA")
plt.show()

RSI

In [ ]:
plt.plot(df['RSI_14'])
plt.title("RSI (14)")
plt.show()

MACD

In [ ]:
plt.plot(df['MACD'])
plt.plot(df['MACD_signal'])
plt.title("MACD & Signal")
plt.show()